In [ ]:
!pip install ipywidgets

In [2]:
import os
from datetime import datetime
from collections import Counter
import ipywidgets as widgets
from IPython.display import display, clear_output

# =========================================================
# SISTEMA DE GESTIÓN HOSPITALARIA - SGH
# Google Colab / Python / ipywidgets
# Incluye persistencia en archivos .txt, citas, registro clínico y análisis de datos
# =========================================================

# =========================================================
# 1. CONFIGURACIÓN INICIAL - CARPETA Y ARCHIVOS
# =========================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = "/content/drive/MyDrive/SGH/datos/"
except Exception:
    RUTA_BASE = "SGH/datos/"

os.makedirs(RUTA_BASE, exist_ok=True)

RUTA_PACIENTES = RUTA_BASE + "pacientes.txt"
RUTA_MEDICOS = RUTA_BASE + "medicos.txt"
RUTA_CITAS = RUTA_BASE + "citas.txt"
RUTA_CONSULTAS = RUTA_BASE + "consultas.txt"
RUTA_TRATAMIENTOS = RUTA_BASE + "tratamientos.txt"
RUTA_MEDICAMENTOS = RUTA_BASE + "medicamentos.txt"

ARCHIVOS = [
    RUTA_PACIENTES,
    RUTA_MEDICOS,
    RUTA_CITAS,
    RUTA_CONSULTAS,
    RUTA_TRATAMIENTOS,
    RUTA_MEDICAMENTOS
]

for archivo in ARCHIVOS:
    if not os.path.exists(archivo):
        with open(archivo, "w", encoding="utf-8") as f:
            pass

# =========================================================
# 2. LISTAS BASE
# =========================================================

TIPOS_SANGRE = ["O+", "O-", "A+", "A-", "B+", "B-", "AB+", "AB-"]
REGIMENES = ["CONTRIBUTIVO", "SUBSIDIADO"]

ESPECIALIDADES = [
    "Medicina General", "Pediatría", "Cardiología",
    "Ortopedia", "Dermatología", "Neurología", "Psiquiatría"
]

CONSULTORIOS = ["A1", "A2", "A3", "B1", "B2", "B3"]
HORARIOS = ["08:00-20:00", "20:00-08:00"]

# =========================================================
# 3. UTILIDADES DE ARCHIVO
# =========================================================

class ArchivoUtil:

    @staticmethod
    def leer_lineas(ruta):
        if not os.path.exists(ruta):
            return []
        with open(ruta, "r", encoding="utf-8") as f:
            return [linea.strip() for linea in f.readlines() if linea.strip()]

    @staticmethod
    def escribir_linea(ruta, linea):
        with open(ruta, "a", encoding="utf-8") as f:
            f.write(linea + "\n")

    @staticmethod
    def reescribir(ruta, lineas):
        with open(ruta, "w", encoding="utf-8") as f:
            for linea in lineas:
                f.write(linea + "\n")

    @staticmethod
    def generar_codigo(prefijo):
        return f"{prefijo}-{datetime.now().strftime('%Y%m%d%H%M%S%f')[:17]}"

# =========================================================
# 4. VALIDACIONES
# =========================================================

def validar_fecha(texto):
    try:
        datetime.strptime(texto, "%Y/%m/%d")
        return True
    except:
        return False

def fecha_a_datetime(texto):
    return datetime.strptime(texto, "%Y/%m/%d")

def texto_limpio(texto):
    return str(texto).replace("|", " ").strip()

def obtener_nombre_paciente(doc):
    for p in cargar_pacientes():
        if p["doc"] == doc:
            return p["nombre"]
    return doc

def obtener_nombre_medico(registro):
    for m in cargar_medicos():
        if m["registro"] == registro:
            return m["nombre"]
    return registro

def obtener_especialidad_medico(registro):
    for m in cargar_medicos():
        if m["registro"] == registro:
            return m["especialidad"]
    return "Sin especialidad"

# =========================================================
# 5. CARGA DE DATOS DESDE ARCHIVOS
# =========================================================

def cargar_pacientes():
    pacientes = []
    for linea in ArchivoUtil.leer_lineas(RUTA_PACIENTES):
        campos = linea.split("|")
        if len(campos) >= 7:
            pacientes.append({
                "doc": campos[0],
                "nombre": campos[1],
                "fecha": campos[2],
                "sangre": campos[3],
                "eps": campos[4],
                "regimen": campos[5],
                "antecedentes": campos[6]
            })
    return pacientes

def cargar_medicos():
    medicos = []
    for linea in ArchivoUtil.leer_lineas(RUTA_MEDICOS):
        campos = linea.split("|")
        if len(campos) >= 5:
            medicos.append({
                "registro": campos[0],
                "nombre": campos[1],
                "especialidad": campos[2],
                "consultorio": campos[3],
                "horario": campos[4]
            })
    return medicos

def cargar_citas():
    citas = []
    for linea in ArchivoUtil.leer_lineas(RUTA_CITAS):
        campos = linea.split("|")
        if len(campos) >= 7:
            citas.append({
                "codigo": campos[0],
                "doc_paciente": campos[1],
                "registro_medico": campos[2],
                "fecha": campos[3],
                "hora": campos[4],
                "motivo": campos[5],
                "estado": campos[6],
                "motivo_cancelacion": campos[7] if len(campos) > 7 else ""
            })
    return citas

def cargar_consultas():
    consultas = []
    for linea in ArchivoUtil.leer_lineas(RUTA_CONSULTAS):
        campos = linea.split("|")
        if len(campos) >= 10:
            consultas.append({
                "codigo": campos[0],
                "codigo_cita": campos[1],
                "doc_paciente": campos[2],
                "registro_medico": campos[3],
                "fecha": campos[4],
                "diagnostico": campos[5],
                "cie10": campos[6],
                "sintomas": campos[7],
                "signos": campos[8],
                "observaciones": campos[9]
            })
    return consultas

def cargar_tratamientos():
    tratamientos = []
    for linea in ArchivoUtil.leer_lineas(RUTA_TRATAMIENTOS):
        campos = linea.split("|")
        if len(campos) >= 4:
            tratamientos.append({
                "codigo": campos[0],
                "codigo_consulta": campos[1],
                "fecha_inicio": campos[2],
                "duracion_dias": campos[3]
            })
    return tratamientos

def cargar_medicamentos():
    medicamentos = []
    for linea in ArchivoUtil.leer_lineas(RUTA_MEDICAMENTOS):
        campos = linea.split("|")
        if len(campos) >= 5:
            medicamentos.append({
                "codigo_tratamiento": campos[0],
                "nombre": campos[1],
                "dosis": campos[2],
                "frecuencia": campos[3],
                "duracion_dias": campos[4]
            })
    return medicamentos

# =========================================================
# 6. FUNCIONES DE GUARDADO
# =========================================================

def guardar_paciente(p):
    linea = "|".join([
        p["doc"], p["nombre"], p["fecha"], p["sangre"],
        p["eps"], p["regimen"], p["antecedentes"]
    ])
    ArchivoUtil.escribir_linea(RUTA_PACIENTES, linea)

def guardar_medico(m):
    linea = "|".join([
        m["registro"], m["nombre"], m["especialidad"],
        m["consultorio"], m["horario"]
    ])
    ArchivoUtil.escribir_linea(RUTA_MEDICOS, linea)

def guardar_cita(c):
    linea = "|".join([
        c["codigo"], c["doc_paciente"], c["registro_medico"],
        c["fecha"], c["hora"], c["motivo"], c["estado"],
        c.get("motivo_cancelacion", "")
    ])
    ArchivoUtil.escribir_linea(RUTA_CITAS, linea)

def guardar_consulta(c):
    linea = "|".join([
        c["codigo"], c["codigo_cita"], c["doc_paciente"],
        c["registro_medico"], c["fecha"], c["diagnostico"],
        c["cie10"], c["sintomas"], c["signos"], c["observaciones"]
    ])
    ArchivoUtil.escribir_linea(RUTA_CONSULTAS, linea)

def guardar_tratamiento(t):
    linea = "|".join([
        t["codigo"], t["codigo_consulta"],
        t["fecha_inicio"], str(t["duracion_dias"])
    ])
    ArchivoUtil.escribir_linea(RUTA_TRATAMIENTOS, linea)

def guardar_medicamento(med):
    linea = "|".join([
        med["codigo_tratamiento"], med["nombre"], med["dosis"],
        med["frecuencia"], str(med["duracion_dias"])
    ])
    ArchivoUtil.escribir_linea(RUTA_MEDICAMENTOS, linea)

# =========================================================
# 7. HORARIOS PARA CITAS
# =========================================================

def generar_horas(horario):
    inicio, fin = horario.split("-")
    h_inicio = int(inicio[:2])
    h_fin = int(fin[:2])
    horas = []
    actual = h_inicio

    while True:
        horas.append(f"{actual:02d}:00")
        horas.append(f"{actual:02d}:30")

        actual += 1
        if actual == 24:
            actual = 0

        if actual == h_fin:
            break

        if len(horas) > 48:
            break

    return horas

# =========================================================
# 8. MENÚ PACIENTES
# =========================================================

def vista_pacientes():
    clear_output()
    display(widgets.HTML("<h2>Gestión de Pacientes</h2>"))

    doc = widgets.Text(description="Documento:")
    nombre = widgets.Text(description="Nombre:")
    fecha = widgets.Text(description="Fecha nac:", placeholder="AAAA/MM/DD")
    sangre = widgets.Dropdown(options=TIPOS_SANGRE, description="Sangre:")
    eps = widgets.Text(description="EPS:")
    regimen = widgets.Dropdown(options=REGIMENES, description="Régimen:")
    antecedentes = widgets.Text(description="Antecedentes:")

    btn_guardar = widgets.Button(description="Guardar paciente", button_style="success")
    btn_listar = widgets.Button(description="Listar pacientes")
    btn_volver = widgets.Button(description="Volver al menú")
    salida = widgets.Output()

    def guardar(b):
        with salida:
            clear_output()
            if not doc.value.strip() or not nombre.value.strip():
                print("Documento y nombre son obligatorios.")
                return

            if not validar_fecha(fecha.value):
                print("Fecha inválida. Use el formato AAAA/MM/DD.")
                return

            existentes = cargar_pacientes()
            if any(p["doc"] == doc.value.strip() for p in existentes):
                print("Ya existe un paciente con ese documento.")
                return

            p = {
                "doc": texto_limpio(doc.value),
                "nombre": texto_limpio(nombre.value),
                "fecha": texto_limpio(fecha.value),
                "sangre": sangre.value,
                "eps": texto_limpio(eps.value),
                "regimen": regimen.value,
                "antecedentes": texto_limpio(antecedentes.value) or "NINGUNO"
            }

            guardar_paciente(p)
            print("Paciente registrado correctamente.")

    def listar(b):
        with salida:
            clear_output()
            pacientes = cargar_pacientes()
            if not pacientes:
                print("No hay pacientes registrados.")
                return
            for p in pacientes:
                print(f"{p['doc']} | {p['nombre']} | {p['fecha']} | {p['sangre']} | {p['eps']} | {p['regimen']} | {p['antecedentes']}")

    btn_guardar.on_click(guardar)
    btn_listar.on_click(listar)
    btn_volver.on_click(lambda b: mostrar_menu())

    display(doc, nombre, fecha, sangre, eps, regimen, antecedentes,
            widgets.HBox([btn_guardar, btn_listar, btn_volver]), salida)

# =========================================================
# 9. MENÚ MÉDICOS
# =========================================================

def vista_medicos():
    clear_output()
    display(widgets.HTML("<h2>Gestión de Médicos</h2>"))

    registro = widgets.Text(description="Registro:")
    nombre = widgets.Text(description="Nombre:")
    especialidad = widgets.Dropdown(options=ESPECIALIDADES, description="Especialidad:")
    consultorio = widgets.Dropdown(options=CONSULTORIOS, description="Consultorio:")
    horario = widgets.Dropdown(options=HORARIOS, description="Horario:")

    btn_guardar = widgets.Button(description="Guardar médico", button_style="success")
    btn_listar = widgets.Button(description="Listar médicos")
    btn_filtrar = widgets.Button(description="Buscar por especialidad")
    btn_volver = widgets.Button(description="Volver al menú")
    salida = widgets.Output()

    def guardar(b):
        with salida:
            clear_output()
            if not registro.value.strip() or not nombre.value.strip():
                print("Registro y nombre son obligatorios.")
                return

            existentes = cargar_medicos()
            if any(m["registro"] == registro.value.strip() for m in existentes):
                print("Ya existe un médico con ese registro.")
                return

            m = {
                "registro": texto_limpio(registro.value),
                "nombre": texto_limpio(nombre.value),
                "especialidad": especialidad.value,
                "consultorio": consultorio.value,
                "horario": horario.value
            }

            guardar_medico(m)
            print("Médico registrado correctamente.")

    def listar(b):
        with salida:
            clear_output()
            medicos = cargar_medicos()
            if not medicos:
                print("No hay médicos registrados.")
                return
            for m in medicos:
                print(f"{m['registro']} | {m['nombre']} | {m['especialidad']} | Consultorio {m['consultorio']} | {m['horario']}")

    def filtrar(b):
        with salida:
            clear_output()
            encontrados = [m for m in cargar_medicos() if m["especialidad"] == especialidad.value]
            if not encontrados:
                print("No hay médicos en esa especialidad.")
                return
            for m in encontrados:
                print(f"{m['registro']} | {m['nombre']} | {m['especialidad']} | {m['consultorio']} | {m['horario']}")

    btn_guardar.on_click(guardar)
    btn_listar.on_click(listar)
    btn_filtrar.on_click(filtrar)
    btn_volver.on_click(lambda b: mostrar_menu())

    display(registro, nombre, especialidad, consultorio, horario,
            widgets.HBox([btn_guardar, btn_listar, btn_filtrar, btn_volver]), salida)

# =========================================================
# 10. MENÚ CITAS
# =========================================================

def vista_citas():
    clear_output()
    display(widgets.HTML("<h2>Gestión de Citas</h2>"))

    pacientes = cargar_pacientes()
    medicos = cargar_medicos()

    if not pacientes or not medicos:
        print("Debe registrar al menos un paciente y un médico antes de crear citas.")
        btn_volver = widgets.Button(description="Volver al menú")
        btn_volver.on_click(lambda b: mostrar_menu())
        display(btn_volver)
        return

    paciente = widgets.Dropdown(
        options=[(f"{p['nombre']} - {p['doc']}", p["doc"]) for p in pacientes],
        description="Paciente:"
    )

    medico = widgets.Dropdown(
        options=[(f"{m['nombre']} - {m['especialidad']}", m["registro"]) for m in medicos],
        description="Médico:"
    )

    fecha = widgets.Text(description="Fecha:", placeholder="AAAA/MM/DD")
    hora = widgets.Dropdown(options=[], description="Hora:")
    motivo = widgets.Text(description="Motivo:")

    btn_crear = widgets.Button(description="Programar cita", button_style="success")
    btn_consultar = widgets.Button(description="Consultar por código")
    btn_cancelar = widgets.Button(description="Cancelar cita")
    btn_listar_rango = widgets.Button(description="Listar por paciente/médico y fechas")
    btn_volver = widgets.Button(description="Volver al menú")
    salida = widgets.Output()

    def actualizar_horas(change=None):
        lista_medicos = cargar_medicos()
        m = next((x for x in lista_medicos if x["registro"] == medico.value), None)
        if m:
            hora.options = generar_horas(m["horario"])

    medico.observe(actualizar_horas, names="value")
    actualizar_horas()

    codigo_busqueda = widgets.Text(description="Código cita:")
    motivo_cancelacion = widgets.Text(description="Motivo canc.:")

    tipo_listado = widgets.Dropdown(options=["Paciente", "Médico"], description="Listar por:")
    selector_listado = widgets.Dropdown(description="Seleccione:")
    fecha_inicio = widgets.Text(description="Fecha inicio:", placeholder="AAAA/MM/DD")
    fecha_fin = widgets.Text(description="Fecha fin:", placeholder="AAAA/MM/DD")

    def actualizar_selector(change=None):
        if tipo_listado.value == "Paciente":
            selector_listado.options = [(f"{p['nombre']} - {p['doc']}", p["doc"]) for p in cargar_pacientes()]
        else:
            selector_listado.options = [(f"{m['nombre']} - {m['registro']}", m["registro"]) for m in cargar_medicos()]

    tipo_listado.observe(actualizar_selector, names="value")
    actualizar_selector()

    def crear(b):
        with salida:
            clear_output()

            if not validar_fecha(fecha.value):
                print("Fecha inválida. Use AAAA/MM/DD.")
                return

            citas = cargar_citas()

            ocupada = any(
                c["registro_medico"] == medico.value and
                c["fecha"] == fecha.value and
                c["hora"] == hora.value and
                c["estado"] == "ACTIVA"
                for c in citas
            )

            if ocupada:
                print("Ese médico ya tiene una cita activa en esa fecha y hora.")
                return

            c = {
                "codigo": ArchivoUtil.generar_codigo("CIT"),
                "doc_paciente": paciente.value,
                "registro_medico": medico.value,
                "fecha": fecha.value,
                "hora": hora.value,
                "motivo": texto_limpio(motivo.value),
                "estado": "ACTIVA",
                "motivo_cancelacion": ""
            }

            guardar_cita(c)
            print("Cita programada correctamente.")
            print("Código:", c["codigo"])

    def consultar(b):
        with salida:
            clear_output()
            cod = codigo_busqueda.value.strip()
            cita = next((c for c in cargar_citas() if c["codigo"] == cod), None)

            if not cita:
                print("No existe una cita con ese código.")
                return

            print("DETALLE DE LA CITA")
            print("Código:", cita["codigo"])
            print("Paciente:", obtener_nombre_paciente(cita["doc_paciente"]))
            print("Médico:", obtener_nombre_medico(cita["registro_medico"]))
            print("Fecha:", cita["fecha"])
            print("Hora:", cita["hora"])
            print("Motivo:", cita["motivo"])
            print("Estado:", cita["estado"])
            if cita["motivo_cancelacion"]:
                print("Motivo de cancelación:", cita["motivo_cancelacion"])

    def cancelar(b):
        with salida:
            clear_output()
            cod = codigo_busqueda.value.strip()
            citas = cargar_citas()
            encontrada = False
            nuevas_lineas = []

            for c in citas:
                if c["codigo"] == cod and c["estado"] == "ACTIVA":
                    c["estado"] = "CANCELADA"
                    c["motivo_cancelacion"] = texto_limpio(motivo_cancelacion.value) or "Sin motivo registrado"
                    encontrada = True

                linea = "|".join([
                    c["codigo"], c["doc_paciente"], c["registro_medico"],
                    c["fecha"], c["hora"], c["motivo"], c["estado"],
                    c.get("motivo_cancelacion", "")
                ])
                nuevas_lineas.append(linea)

            ArchivoUtil.reescribir(RUTA_CITAS, nuevas_lineas)

            if encontrada:
                print("Cita cancelada correctamente.")
            else:
                print("No se encontró una cita activa con ese código.")

    def listar_rango(b):
        with salida:
            clear_output()

            if not validar_fecha(fecha_inicio.value) or not validar_fecha(fecha_fin.value):
                print("Fechas inválidas. Use AAAA/MM/DD.")
                return

            inicio = fecha_a_datetime(fecha_inicio.value)
            fin = fecha_a_datetime(fecha_fin.value)

            if inicio > fin:
                print("La fecha inicial no puede ser mayor que la fecha final.")
                return

            encontrados = []

            for c in cargar_citas():
                fecha_c = fecha_a_datetime(c["fecha"])

                if tipo_listado.value == "Paciente":
                    coincide = c["doc_paciente"] == selector_listado.value
                else:
                    coincide = c["registro_medico"] == selector_listado.value

                if coincide and inicio <= fecha_c <= fin:
                    encontrados.append(c)

            if not encontrados:
                print("No hay citas en ese rango de fechas.")
                return

            print(f"CITAS POR {tipo_listado.value.upper()} ENTRE {fecha_inicio.value} Y {fecha_fin.value}")
            print("-" * 80)
            for c in sorted(encontrados, key=lambda x: (x["fecha"], x["hora"])):
                print(f"{c['codigo']} | {c['fecha']} {c['hora']} | Paciente: {obtener_nombre_paciente(c['doc_paciente'])} | Médico: {obtener_nombre_medico(c['registro_medico'])} | Estado: {c['estado']} | Motivo: {c['motivo']}")

    btn_crear.on_click(crear)
    btn_consultar.on_click(consultar)
    btn_cancelar.on_click(cancelar)
    btn_listar_rango.on_click(listar_rango)
    btn_volver.on_click(lambda b: mostrar_menu())

    display(
        widgets.HTML("<h3>Programar cita</h3>"),
        paciente, medico, fecha, hora, motivo,
        widgets.HBox([btn_crear, btn_volver]),

        widgets.HTML("<h3>Consultar o cancelar cita</h3>"),
        codigo_busqueda, motivo_cancelacion,
        widgets.HBox([btn_consultar, btn_cancelar]),

        widgets.HTML("<h3>Listar citas por paciente o médico en rango de fechas</h3>"),
        tipo_listado, selector_listado, fecha_inicio, fecha_fin,
        btn_listar_rango,
        salida
    )

# =========================================================
# 11. REGISTRO CLÍNICO
# =========================================================

def vista_registro_clinico():
    clear_output()
    display(widgets.HTML("<h2>Registro Clínico</h2>"))

    pacientes = cargar_pacientes()
    medicos = cargar_medicos()
    citas = [c for c in cargar_citas() if c["estado"] == "ACTIVA"]

    if not pacientes or not medicos:
        print("Debe registrar pacientes y médicos primero.")
        btn_volver = widgets.Button(description="Volver al menú")
        btn_volver.on_click(lambda b: mostrar_menu())
        display(btn_volver)
        return

    cita = widgets.Dropdown(
        options=[(f"{c['codigo']} | {obtener_nombre_paciente(c['doc_paciente'])} | {c['fecha']} {c['hora']}", c["codigo"]) for c in citas] or [("Sin cita asociada", "SIN_CITA")],
        description="Cita:"
    )

    paciente = widgets.Dropdown(
        options=[(f"{p['nombre']} - {p['doc']}", p["doc"]) for p in pacientes],
        description="Paciente:"
    )

    medico = widgets.Dropdown(
        options=[(f"{m['nombre']} - {m['especialidad']}", m["registro"]) for m in medicos],
        description="Médico:"
    )

    diagnostico = widgets.Text(description="Diagnóstico:")
    cie10 = widgets.Text(description="CIE-10:", placeholder="Ej: J00")
    sintomas = widgets.Text(description="Síntomas:")
    presion = widgets.Text(description="Presión:")
    temperatura = widgets.Text(description="Temperatura:")
    frecuencia = widgets.Text(description="F. cardíaca:")
    observaciones = widgets.Text(description="Observaciones:")

    medicamento = widgets.Text(description="Medicamento:")
    dosis = widgets.Text(description="Dosis:")
    frecuencia_med = widgets.Text(description="Frecuencia:")
    duracion = widgets.IntText(description="Días:", value=1)

    btn_guardar = widgets.Button(description="Guardar consulta y tratamiento", button_style="success")
    btn_historial = widgets.Button(description="Consultar historial")
    btn_volver = widgets.Button(description="Volver al menú")
    salida = widgets.Output()

    def cargar_datos_cita(change=None):
        if cita.value == "SIN_CITA":
            return
        c = next((x for x in cargar_citas() if x["codigo"] == cita.value), None)
        if c:
            paciente.value = c["doc_paciente"]
            medico.value = c["registro_medico"]

    cita.observe(cargar_datos_cita, names="value")
    cargar_datos_cita()

    def guardar(b):
        with salida:
            clear_output()

            if not diagnostico.value.strip() or not cie10.value.strip():
                print("Debe ingresar diagnóstico y código CIE-10.")
                return

            codigo_consulta = ArchivoUtil.generar_codigo("CON")
            codigo_tratamiento = ArchivoUtil.generar_codigo("TRA")

            signos = f"Presión:{texto_limpio(presion.value)}, Temperatura:{texto_limpio(temperatura.value)}, FC:{texto_limpio(frecuencia.value)}"

            consulta = {
                "codigo": codigo_consulta,
                "codigo_cita": cita.value,
                "doc_paciente": paciente.value,
                "registro_medico": medico.value,
                "fecha": datetime.now().strftime("%Y/%m/%d"),
                "diagnostico": texto_limpio(diagnostico.value),
                "cie10": texto_limpio(cie10.value).upper(),
                "sintomas": texto_limpio(sintomas.value),
                "signos": signos,
                "observaciones": texto_limpio(observaciones.value)
            }

            tratamiento = {
                "codigo": codigo_tratamiento,
                "codigo_consulta": codigo_consulta,
                "fecha_inicio": datetime.now().strftime("%Y/%m/%d"),
                "duracion_dias": duracion.value
            }

            med = {
                "codigo_tratamiento": codigo_tratamiento,
                "nombre": texto_limpio(medicamento.value),
                "dosis": texto_limpio(dosis.value),
                "frecuencia": texto_limpio(frecuencia_med.value),
                "duracion_dias": duracion.value
            }

            guardar_consulta(consulta)
            guardar_tratamiento(tratamiento)

            if medicamento.value.strip():
                guardar_medicamento(med)

            # Marcar la cita como ATENDIDA
            if cita.value != "SIN_CITA":
                nuevas_lineas = []
                for c in cargar_citas():
                    if c["codigo"] == cita.value:
                        c["estado"] = "ATENDIDA"

                    nuevas_lineas.append("|".join([
                        c["codigo"], c["doc_paciente"], c["registro_medico"],
                        c["fecha"], c["hora"], c["motivo"], c["estado"],
                        c.get("motivo_cancelacion", "")
                    ]))
                ArchivoUtil.reescribir(RUTA_CITAS, nuevas_lineas)

            print("Consulta, diagnóstico y tratamiento registrados correctamente.")
            print("Código de consulta:", codigo_consulta)

    def historial(b):
        with salida:
            clear_output()

            historial = [c for c in cargar_consultas() if c["doc_paciente"] == paciente.value]
            historial.sort(key=lambda x: x["fecha"])

            if not historial:
                print("El paciente no tiene historial clínico registrado.")
                return

            print("HISTORIAL CLÍNICO EN ORDEN CRONOLÓGICO")
            print("Paciente:", obtener_nombre_paciente(paciente.value))
            print("-" * 80)

            for c in historial:
                print("Fecha:", c["fecha"])
                print("Médico:", obtener_nombre_medico(c["registro_medico"]))
                print("Diagnóstico:", c["diagnostico"], "| CIE-10:", c["cie10"])
                print("Síntomas:", c["sintomas"])
                print("Signos vitales:", c["signos"])
                print("Observaciones:", c["observaciones"])

                tratamientos = [t for t in cargar_tratamientos() if t["codigo_consulta"] == c["codigo"]]
                for t in tratamientos:
                    meds = [m for m in cargar_medicamentos() if m["codigo_tratamiento"] == t["codigo"]]
                    for med in meds:
                        print("Medicamento:", med["nombre"], "| Dosis:", med["dosis"], "| Frecuencia:", med["frecuencia"], "| Días:", med["duracion_dias"])
                print("-" * 80)

    btn_guardar.on_click(guardar)
    btn_historial.on_click(historial)
    btn_volver.on_click(lambda b: mostrar_menu())

    display(
        widgets.HTML("<h3>Datos de la consulta</h3>"),
        cita, paciente, medico,
        diagnostico, cie10, sintomas, presion, temperatura, frecuencia, observaciones,
        widgets.HTML("<h3>Tratamiento</h3>"),
        medicamento, dosis, frecuencia_med, duracion,
        widgets.HBox([btn_guardar, btn_historial, btn_volver]),
        salida
    )

# =========================================================
# 12. ANÁLISIS DE DATOS COMPLETO
# =========================================================

def vista_analisis():
    clear_output()
    display(widgets.HTML("<h2>Análisis de Datos</h2>"))

    fecha_inicio = widgets.Text(description="Fecha inicio:", placeholder="AAAA/MM/DD")
    fecha_fin = widgets.Text(description="Fecha fin:", placeholder="AAAA/MM/DD")
    top_n = widgets.IntText(description="Top N:", value=5)

    btn_promedio = widgets.Button(description="1. Promedio consultas por médico")
    btn_diag = widgets.Button(description="2. Diagnósticos frecuentes CIE-10")
    btn_ocupacion = widgets.Button(description="3. Ocupación por especialidad")
    btn_eps = widgets.Button(description="4. Distribución EPS y régimen")
    btn_meds = widgets.Button(description="5. Medicamentos más prescritos")
    btn_volver = widgets.Button(description="Volver al menú")
    salida = widgets.Output()

    def filtrar_por_fecha(lista, campo_fecha="fecha"):
        if not validar_fecha(fecha_inicio.value) or not validar_fecha(fecha_fin.value):
            return None

        inicio = fecha_a_datetime(fecha_inicio.value)
        fin = fecha_a_datetime(fecha_fin.value)

        if inicio > fin:
            return []

        return [
            item for item in lista
            if inicio <= fecha_a_datetime(item[campo_fecha]) <= fin
        ]

    def promedio_consultas(b):
        with salida:
            clear_output()

            consultas = filtrar_por_fecha(cargar_consultas())

            if consultas is None:
                print("Ingrese fechas válidas en formato AAAA/MM/DD.")
                return

            if not consultas:
                print("No hay consultas en ese periodo.")
                return

            dias_periodo = (fecha_a_datetime(fecha_fin.value) - fecha_a_datetime(fecha_inicio.value)).days + 1
            conteo = Counter(c["registro_medico"] for c in consultas)

            print("PROMEDIO DE CONSULTAS POR MÉDICO")
            print(f"Periodo: {fecha_inicio.value} a {fecha_fin.value}")
            print("-" * 80)

            for registro, total in conteo.items():
                promedio = total / dias_periodo
                print(f"Médico: {obtener_nombre_medico(registro)} | Total consultas: {total} | Promedio diario: {promedio:.2f}")

    def diagnosticos_frecuentes(b):
        with salida:
            clear_output()

            consultas = cargar_consultas()
            if fecha_inicio.value.strip() and fecha_fin.value.strip():
                consultas = filtrar_por_fecha(consultas)
                if consultas is None:
                    print("Ingrese fechas válidas en formato AAAA/MM/DD.")
                    return

            if not consultas:
                print("No hay consultas registradas.")
                return

            contador = Counter(c["cie10"] for c in consultas)

            print("DIAGNÓSTICOS MÁS FRECUENTES POR CÓDIGO CIE-10")
            print("-" * 80)

            for cie, cantidad in contador.most_common(top_n.value):
                descripcion = next((c["diagnostico"] for c in consultas if c["cie10"] == cie), "")
                print(f"{cie} | {descripcion} | {cantidad} veces")

    def ocupacion_especialidad(b):
        with salida:
            clear_output()

            citas = cargar_citas()
            if fecha_inicio.value.strip() and fecha_fin.value.strip():
                citas = filtrar_por_fecha(citas)
                if citas is None:
                    print("Ingrese fechas válidas en formato AAAA/MM/DD.")
                    return

            if not citas:
                print("No hay citas registradas en el periodo.")
                return

            total_por_esp = Counter()
            atendidas_por_esp = Counter()

            for c in citas:
                esp = obtener_especialidad_medico(c["registro_medico"])
                total_por_esp[esp] += 1
                if c["estado"] == "ATENDIDA":
                    atendidas_por_esp[esp] += 1

            print("TASA DE OCUPACIÓN POR ESPECIALIDAD")
            print("-" * 80)
            print("Especialidad | Citas programadas | Citas atendidas | Tasa (%)")
            print("-" * 80)

            for esp, total in total_por_esp.items():
                atendidas = atendidas_por_esp[esp]
                tasa = (atendidas / total) * 100 if total > 0 else 0
                print(f"{esp} | {total} | {atendidas} | {tasa:.2f}%")

    def distribucion_eps(b):
        with salida:
            clear_output()

            pacientes = cargar_pacientes()
            if not pacientes:
                print("No hay pacientes registrados.")
                return

            conteo_eps = Counter(p["eps"] for p in pacientes)
            conteo_regimen = Counter(p["regimen"] for p in pacientes)
            conteo_combinado = Counter((p["eps"], p["regimen"]) for p in pacientes)

            print("DISTRIBUCIÓN DE PACIENTES POR EPS")
            print("-" * 80)
            for eps, cantidad in conteo_eps.most_common():
                porcentaje = (cantidad / len(pacientes)) * 100
                print(f"{eps}: {cantidad} pacientes ({porcentaje:.2f}%)")

            print("\nDISTRIBUCIÓN POR RÉGIMEN")
            print("-" * 80)
            for regimen, cantidad in conteo_regimen.most_common():
                porcentaje = (cantidad / len(pacientes)) * 100
                print(f"{regimen}: {cantidad} pacientes ({porcentaje:.2f}%)")

            print("\nDISTRIBUCIÓN COMBINADA EPS + RÉGIMEN")
            print("-" * 80)
            for (eps, regimen), cantidad in conteo_combinado.most_common():
                porcentaje = (cantidad / len(pacientes)) * 100
                print(f"{eps} | {regimen}: {cantidad} pacientes ({porcentaje:.2f}%)")

    def medicamentos_prescritos(b):
        with salida:
            clear_output()

            medicamentos = cargar_medicamentos()
            if not medicamentos:
                print("No hay medicamentos registrados.")
                return

            contador = Counter(m["nombre"].upper() for m in medicamentos if m["nombre"].strip())

            print("MEDICAMENTOS MÁS PRESCRITOS")
            print("-" * 80)

            for nombre, cantidad in contador.most_common(top_n.value):
                print(f"{nombre}: {cantidad} veces")

    btn_promedio.on_click(promedio_consultas)
    btn_diag.on_click(diagnosticos_frecuentes)
    btn_ocupacion.on_click(ocupacion_especialidad)
    btn_eps.on_click(distribucion_eps)
    btn_meds.on_click(medicamentos_prescritos)
    btn_volver.on_click(lambda b: mostrar_menu())

    display(
        widgets.HTML("<p>Para los reportes por periodo, use formato AAAA/MM/DD.</p>"),
        fecha_inicio, fecha_fin, top_n,
        btn_promedio, btn_diag, btn_ocupacion, btn_eps, btn_meds,
        btn_volver,
        salida
    )

# =========================================================
# 13. MENÚ PRINCIPAL
# =========================================================

menu = widgets.Dropdown(
    options=[
        ("Pacientes", 1),
        ("Médicos", 2),
        ("Citas", 3),
        ("Registro Clínico", 4),
        ("Análisis de Datos", 5)
    ],
    description="Módulo:"
)

btn_ir = widgets.Button(description="Ir", button_style="primary")

def navegar(b):
    if menu.value == 1:
        vista_pacientes()
    elif menu.value == 2:
        vista_medicos()
    elif menu.value == 3:
        vista_citas()
    elif menu.value == 4:
        vista_registro_clinico()
    elif menu.value == 5:
        vista_analisis()

btn_ir.on_click(navegar)

def mostrar_menu():
    clear_output()
    display(widgets.HTML("<h1>Sistema de Gestión Hospitalaria</h1>"))
    display(widgets.HTML(f"<p><b>Ruta de datos:</b> {RUTA_BASE}</p>"))
    display(menu, btn_ir)

mostrar_menu()


HTML(value='<h1>Sistema de Gestión Hospitalaria</h1>')

HTML(value='<p><b>Ruta de datos:</b> /content/drive/MyDrive/SGH/datos/</p>')

Dropdown(description='Módulo:', index=4, options=(('Pacientes', 1), ('Médicos', 2), ('Citas', 3), ('Registro C…

Button(button_style='primary', description='Ir', style=ButtonStyle())